# Lab 5: Whisper STT Implementation

## Step 1: Setting up the Environment

In [1]:
# pip install openai pydub

In [2]:
import sounddevice as sd
import numpy as np
# import librosa
# import librosa.display
import matplotlib.pyplot as plt
from IPython.display import Audio, display
import scipy.io.wavfile as wavfile
from openai import OpenAI
import io
import os
from dotenv import load_dotenv
from pydub import AudioSegment

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

audio_sample = "HI009clip.mp3"


## Step 2 + 3a: Basic Transcription (Without Chunking) - Short part of the file

In [3]:
# Load mp3
audio = AudioSegment.from_mp3(audio_sample)

# Take first 10 seconds (pydub uses milliseconds)
first_10_sec = audio[:10_000]

# Export to in-memory buffer
buffer = io.BytesIO()
first_10_sec.export(buffer, format="mp3")
buffer.seek(0)
buffer.name = audio_sample[:-4] + "_first_10_sec.mp3"

# Transcribe with Whisper
print("🤖 Transcribing with Whisper...")
transcript = client.audio.transcriptions.create(
    model = "whisper-1",
    file = buffer
)

print("\n📝 Transcription:")
print("-" * 40)
print(transcript.text)

🤖 Transcribing with Whisper...

📝 Transcription:
----------------------------------------
Uh, I hear that there's a Hawaiian custom of something called a hukilau. Hukilau, hukilau is a dance.


## Step 2 + 3b/5: Basic Transcription (Without Chunking or Prompts) - Whole file

In [4]:
# Load sample
audio_file = io.BytesIO(open(audio_sample, "rb").read())
audio_file.name = audio_sample

transcript = client.audio.transcriptions.create(
    model="whisper-1",
    file=audio_file
)

print("\n📝 Transcription:")
print("-" * 40)
print(transcript.text)


📝 Transcription:
----------------------------------------
I hear that there's a Hawaiian custom of something called a hukilau. Hukilau, hukilau is they have a rope, end of the rope they have a net there. They surround that certain place and they drag the net. They surround it, you mean they carry the net out? Carry the net, uh-huh. How do they, is there something to keep the net floating? Yeah, some have those floaters floating. Some they have laid down on the bottom and they drag the net in. What do the floats, what are the floats made of? The float made of those, some coconut, dry coconut I guess. Oh, dry coconut, that's a use for coconut. And that floats the top of the net? Top of the net. Now, the word lau means leaf, doesn't it? Lau, lau is a leaf. Alright, lau, where does the leaf come in in this hukilau? Hukilau, the leaves, you know, they sit on the top, top of the rope. So that when they pull, the fish are going to not jump over that rope. The ropes, the leaves keep the fish 

## Step 4: Transcription with Prompts

In [5]:
# Transcribe with context prompt
print("🤖 Transcribing with context...")
transcript = client.audio.transcriptions.create(
    model="whisper-1",
    file=audio_file,
    prompt = "hukilau, lau, akule, Polynese, Japanese, hajiji")

print("\n📝 Transcription:")
print("-" * 40)
print(transcript.text)

🤖 Transcribing with context...

📝 Transcription:
----------------------------------------
I hear that there's a Hawaiian custom of something called a hukilau. Hukilau, hukilau is they have a rope, end of the rope they have a net there. They surround that certain place and they drag the net. They surround it, you mean they carry the net out? Carry the net, uh-huh. How do they, is there something to keep the net floating? Yeah, some have those floaters floating. Some they have laid down on the bottom and they drag the net in. What do the floats, what are the floats made of? The float made of those, some coconut, dry coconut I guess. Oh, dry coconut, that's a use for coconut. And that floats the top of the net? Top of the net. Now, the word lau means leaf, doesn't it? Lau, lau is a leaf. Alright, lau, where does the leaf come in in this hukilau? Hukilau, the leaves, you know, they sit on the top, top of the rope. So that when they pull, the fish are going to not jump over that rope. The r

## Step 6: Chunking

In [6]:
# Load mp3
audio = AudioSegment.from_mp3(audio_sample)

# Split audio into chunks
chunk_length = 5 # seconds
chunk_size = chunk_length * 1000 # miliseconds / positions
chunks = [audio[i:i+chunk_size] for i in range(0, len(audio), chunk_size)]
print(f"\n🔪 Split into {len(chunks)} chunks")


🔪 Split into 37 chunks


In [7]:
# Transcribe each chunk
all_transcripts = []

for i, chunk in enumerate(chunks):
    # Export to in-memory buffer
    buffer = io.BytesIO()
    chunk.export(buffer, format="mp3")
    buffer.seek(0)
    buffer.name = "chunk_{i+1}.mp3"
    
    print(f"\n🤖 Transcribing chunk {i+1}/{len(chunks)}...")   
    # Transcribe
    transcript = client.audio.transcriptions.create(
        model="whisper-1",
        file=buffer
    )
    
    all_transcripts.append(transcript.text)
    print(f"Chunk {i+1}: {transcript.text}")


🤖 Transcribing chunk 1/37...
Chunk 1: Yeah, I hear that there's a Hawaiian custom

🤖 Transcribing chunk 2/37...
Chunk 2: something called a hukilau. Hukilau, hukilau is a

🤖 Transcribing chunk 3/37...
Chunk 3: they have a roof, end of the roof they have a net there, they surround that.

🤖 Transcribing chunk 4/37...
Chunk 4: a certain place and they drag the net. They surround it.

🤖 Transcribing chunk 5/37...
Chunk 5: You mean they carry the net out? Carry the net out. How do they...

🤖 Transcribing chunk 6/37...
Chunk 6: ada sesuatu untuk mengekalkan kapal terbang? ada beberapa kapal terbang yang terbang

🤖 Transcribing chunk 7/37...
Chunk 7: Some of them were laid down on the bottom and they dragged the net in.

🤖 Transcribing chunk 8/37...
Chunk 8: What do the floats, what are the floats made of? The float made of those, uh...

🤖 Transcribing chunk 9/37...
Chunk 9: Some coconut, dry coconut I guess. Oh, dry coconut.

🤖 Transcribing chunk 10/37...
Chunk 10: Yeah. That's a two-strip 

In [8]:
# Combine all transcripts
print("\n📝 Complete Transcription:")
print("-" * 40)
full_text = " ".join(all_transcripts)
print(full_text)


📝 Complete Transcription:
----------------------------------------
Yeah, I hear that there's a Hawaiian custom something called a hukilau. Hukilau, hukilau is a they have a roof, end of the roof they have a net there, they surround that. a certain place and they drag the net. They surround it. You mean they carry the net out? Carry the net out. How do they... ada sesuatu untuk mengekalkan kapal terbang? ada beberapa kapal terbang yang terbang Some of them were laid down on the bottom and they dragged the net in. What do the floats, what are the floats made of? The float made of those, uh... Some coconut, dry coconut I guess. Oh, dry coconut. Yeah. That's a two-strip coconut. And that floats the top of the nail. Couple. on that. Now the word Lao means leaf doesn't it? Lao is a leaf. It's a leaf. Alright. Now, where does the leaf come in, in this world? Okay, a lot of the leaves, you know, this is one of the top, top of the root. so that when they pull, the fish is not going to jump ove

## Step 7: Transcribing Chunks with Timestamps

In [9]:
# Transcribe each chunk
all_segments = []

for i, chunk in enumerate(chunks):
    chunk_offset_seconds = i * chunk_length
    
    # Export to in-memory buffer
    buffer = io.BytesIO()
    chunk.export(buffer, format="mp3")
    buffer.seek(0)
    buffer.name = "chunk_{i+1}.mp3"
    
    print(f"\n🤖 Transcribing chunk {i+1}/{len(chunks)} with timestamps...")   
    # Transcribe
    transcript = client.audio.transcriptions.create(
        model="whisper-1",
        file=buffer,
        language="en",
        response_format="verbose_json",
        timestamp_granularities=["segment"]  # Get both segment and word timestamps
    )

    for segment in transcript.segments:
        all_segments.append({
            "start": segment.start + chunk_offset_seconds,
            "end": segment.end + chunk_offset_seconds,
            "text": segment.text.strip()
        })


🤖 Transcribing chunk 1/37 with timestamps...

🤖 Transcribing chunk 2/37 with timestamps...

🤖 Transcribing chunk 3/37 with timestamps...

🤖 Transcribing chunk 4/37 with timestamps...

🤖 Transcribing chunk 5/37 with timestamps...

🤖 Transcribing chunk 6/37 with timestamps...

🤖 Transcribing chunk 7/37 with timestamps...

🤖 Transcribing chunk 8/37 with timestamps...

🤖 Transcribing chunk 9/37 with timestamps...

🤖 Transcribing chunk 10/37 with timestamps...

🤖 Transcribing chunk 11/37 with timestamps...

🤖 Transcribing chunk 12/37 with timestamps...

🤖 Transcribing chunk 13/37 with timestamps...

🤖 Transcribing chunk 14/37 with timestamps...

🤖 Transcribing chunk 15/37 with timestamps...

🤖 Transcribing chunk 16/37 with timestamps...

🤖 Transcribing chunk 17/37 with timestamps...

🤖 Transcribing chunk 18/37 with timestamps...

🤖 Transcribing chunk 19/37 with timestamps...

🤖 Transcribing chunk 20/37 with timestamps...

🤖 Transcribing chunk 21/37 with timestamps...

🤖 Transcribing chunk 

In [10]:
# Print segments
for seg in all_segments:
    print(f"[{seg['start']:.2f} - {seg['end']:.2f}] {seg['text']}")

# Combine full text
full_text = " ".join(seg["text"] for seg in all_segments)

print("\n:memo: Complete Transcription:")
print("-" * 40)
print(full_text)

[0.00 - 4.20] Yeah, I hear that there's a Hawaiian custom
[5.00 - 11.88] something called a hukilau. Hukilau, hukilau is a
[10.00 - 15.00] they have a roof, end of the roof they have a net there, they surround that.
[15.00 - 21.96] a certain place and they drag the net. They surround it.
[20.00 - 22.00] You mean they carry the net out?
[22.00 - 24.00] Carry the net out.
[24.00 - 26.00] How do they...
[25.00 - 27.00] Something to keep the net floating?
[27.00 - 30.00] Yeah, sometimes those floaters are floating.
[30.00 - 34.00] Some of them were laid down on the bottom and they dragged the net in.
[35.00 - 37.72] What do the floats, what are the floats made of?
[37.72 - 39.96] The float made of those, uh...
[40.00 - 44.00] Some coconut, dry coconut I guess.
[44.00 - 45.00] Oh, dry coconut.
[45.00 - 45.50] Yeah.
[45.50 - 48.20] That's a two-strip coconut.
[48.20 - 49.80] And that floats the top of the nail.
[49.80 - 51.34] Couple.
[50.00 - 57.00] on that. Now the word Lao means leaf does

## Step 8: Exporting with Timestamps

In [11]:
# Export plain text with timestamps
with open("transcription_with_timestamps.txt", "w", encoding="utf-8") as f:
    for seg in all_segments:
        f.write(f"[{seg['start']:.2f} - {seg['end']:.2f}] {seg['text']}\n")

In [13]:
def format_timestamp(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    millis = int((seconds - int(seconds)) * 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"

with open("transcription.srt", "w", encoding="utf-8") as f:
    for i, seg in enumerate(all_segments, start=1):
        start = format_timestamp(seg['start'])
        end = format_timestamp(seg['end'])
        text = seg['text'].strip()

        f.write(f"{i}\n")
        f.write(f"{start} --> {end}\n")
        f.write(f"{text}\n\n")

In [12]:
# Export JSON
import json
with open("transcription_with_timestamps.json", "w", encoding="utf-8") as f:
    json.dump(all_segments, f, indent=2, ensure_ascii=False)